# First Modal app: default vs. custom container and logging

Two functions on the same app: `f_default` runs in Modal's stock container image, `f_custom` runs in an image we build ourselves with an extra dependency (`numpy`).

In [ ]:
import logging
import modal

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

app = modal.App("first-app")


@app.function()
def f_default():
    print("running in a default modal container")
    logger.info("here is a python log")


python_packages = [
    "numpy",
]
custom_container = modal.Image.debian_slim().uv_pip_install(python_packages)

volume = modal.Volume.from_name("my-volume", create_if_missing=True)


# in Modal, each function runs as a separate python process with its own environment.
# this environment is specified by the image and volumes will appear as folders defined
# with respect to root of the filesystem. So from this function's perspective, the volume
# lives as a folder /store. Some other function might define it differently and for that
# container (isolated linux environment) the same volume might appear in a different path.
@app.function(image=custom_container, volumes={"/store": volume})
def f_custom(x):
    # logging.basicConfig(level=logging.INFO)
    # logger = logging.getLogger(__name__)
    from pathlib import Path
    import numpy as np

    ans = np.sqrt(x)
    print(f"sqrt {x} = {ans}")
    out_dir = Path("/store/test")
    out_dir.mkdir(parents=True, exist_ok=True)
    np.save(out_dir / f"answer_to_square_of_{x}.npy", ans)

**Incident note (2026-07-22):** `f_custom` originally called `np.save(f"/store/test/answer_to_square_of_{x}.npy", ans)` without first creating `/store/test` on the mounted Volume. Instead of failing fast with a `FileNotFoundError` (as a normal local filesystem would), the write hung and the run had to be aborted manually. Root cause wasn't confirmed against a specific filed Modal bug — Volumes are FUSE-backed over gRPC, and it's plausible the "create" RPC doesn't reject cleanly when the parent directory doesn't exist. Fix applied above: explicitly `Path("/store/test").mkdir(parents=True, exist_ok=True)` before writing.

A few things happening in the cell above:

- **`f_default`** gets no `image=` argument, so it runs in Modal's default container — a plain Debian + Python image with no extra packages.
- **`custom_container`** is a `modal.Image`, which represents an actual Docker image. `modal.Image.debian_slim()` picks Modal's minimal Debian base, and `.uv_pip_install(python_packages)` adds a build step that installs the given Python packages into that image using `uv` (Modal builds and caches this image once, then reuses it for every container that runs `f_custom`).
- **`f_custom`** passes `image=custom_container`, so its container is built from that image instead of the default — that's why it's able to `import numpy` even though `numpy` was never installed in this notebook's own environment.
- The `logging.basicConfig(...)` / `logger` setup at the top only configures logging in *this* notebook process. It does not travel with the function to the remote container — more on that below.

In [ ]:
with modal.enable_output():
    with app.run():
        f_custom.remote(2)
        f_custom.remote(12)

Notice the output only shows `running in a default modal container` (the `print(...)` call) — `here is a python log` (the `logger.info(...)` call) never shows up.

That's because `f_default` runs in a brand-new Python process inside the remote container. Modal ships the function's *code* there (via cloudpickle), but not the side effects of module-level statements that already ran locally — so `logging.basicConfig(level=logging.INFO)` never executes in that container. Without it, the root logger in the remote process defaults to level `WARNING` with no handler attached, so an `INFO` record is silently dropped. `print()` is unaffected by any of this — it always writes to stdout, which Modal captures and streams back whenever `modal.enable_output()` is active.

Fix: call `logging.basicConfig(...)` *inside* the function body (so it runs remotely too), e.g.:

```python
@app.function()
def f_default():
    logging.basicConfig(level=logging.INFO)
    logger = logging.getLogger(__name__)
    print('running in a default modal container')
    logger.info('here is a python log')
```

The two commented-out lines at the top of `f_custom` (`# logging.basicConfig(...)` / `# logger = logging.getLogger(__name__)`) are that exact same fix, left in place but inactive. `f_custom` doesn't currently call `logger.info(...)` anywhere — it only uses `print(...)`, which works remotely regardless of logging config — so those lines aren't doing anything right now, commented or not. If you later add `logger.info(...)` calls inside `f_custom` and want them to actually show up in the captured output, uncomment both lines (just like in `f_default` above); until then they're just a placeholder reminder of the pattern.

It is possible to spawn a container and attach a volume to explore files in shell
```bash[]
uv run modal shell --volume my-volume
```
These CLI commands also allow you to transfer files between local and remote

In [ ]:
from pathlib import Path
import modal

project_name = "language-model-pretraining"
LOCAL_STORAGE_ROOT = Path("/Users/oguz/Projects/transformer/volume")
app = modal.App(project_name)

container = (
    modal.Image.debian_slim(python_version="3.12")
    .uv_sync(uv_project_dir="/Users/oguz/Projects/transformer")
    .add_local_python_source("transformer")
)

volume = modal.Volume.from_name("language-model-pretraining", create_if_missing=True)


@app.cls(image=container, volumes={"/storage": volume})
class Pipeline:
    @modal.enter()
    def setup(self):
        self.storage_root = Path("/storage") if not modal.is_local() else LOCAL_STORAGE_ROOT
        print(self.storage_root)

    @modal.method()
    def show(self):
        for f in (self.storage_root / "data").iterdir():
            print(f)


# with modal.enable_output():
with app.run():
    Pipeline().show.remote()
with app.run():
    Pipeline().show.local()